In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool
from langsmith import traceable

import requests
import random
import asyncio

C:\Users\sarth\AppData\Local\Temp\ipykernel_39944\987210721.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [2]:
load_dotenv()
import os

api_key = os.getenv("GOOGLE_API_KEY")

In [3]:
llm =  ChatGoogleGenerativeAI(
    model = 'gemini-3.8-flash',
    api_key = api_key
)

In [4]:
# TOOLS

search_tool = DuckDuckGoSearchRun(region='us-en')

@tool
@traceable
def calculator(first_num: float, second_num: float, operation: str) -> dict:
    """ Perform a basic arithmetic operation on two numbers."""

    try: 
        if operation == 'add':
            result = first_num + second_num
        elif operation == 'subtract':
            result = first_num - second_num
        elif operation == 'multiply':
            result = first_num * second_num
        elif operation == 'divide':
            result = first_num / second_num
        else:
            return {'error': f"Unsupported operation: {operation}. Supported operations are add, subtract, multiply, divide."}
        return {'first_num': first_num, 'second_num': second_num, 'operation': operation, 'result': result}
    except ZeroDivisionError:
        return {'error': 'Division by zero is not allowed.'}
    except Exception as e:
        return {'error': str(e)}



In [5]:
tools = [search_tool, calculator]

llm_with_tools = llm.bind_tools(tools)

In [6]:
class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
def build_graph():

    async def chat_node(state: ChatState):
        messages = state['messages']
        response = await llm_with_tools.ainvoke(messages)
        return {'messages':[response]}
    
    tool_node = ToolNode(tools)
    graph = StateGraph(ChatState)
    graph.add_node('chat_node', chat_node) 
    graph.add_node('tools', tool_node)
    graph.add_edge(START, 'chat_node')
    graph.add_conditional_edges("chat_node", tools_condition)
    graph.add_edge('tools', 'chat_node')
    chatbot = graph.compile()
    return chatbot
 

async def main():

    chatbot = build_graph()

    out = await chatbot.ainvoke({'messages': [HumanMessage(content="what is the current cricket headline today? calculate 40 and 30 sum?")]})
print(out['messages'][-1].content)

if __name__ == "__main__":
    asyncio.run(main())